# 07 — Tính AFELO bằng Cardiff sentiment và hoàn thiện nhãn cuối kỳ

Đọc bảng comment đã lọc từ notebook 06, chạy `cardiffnlp/twitter-roberta-base-sentiment-latest` tại máy và lưu ba xác suất theo comment_id. Sau đó điền AFELO_proxy, LOQV từng chương và q/y từng user–course. Không quét lại video/problem.

Model tiếng Anh, được pin bằng revision và checksum. Đây là sentiment trên comment dịch trong miền giáo dục; cần kiểm định thủ công trước khi diễn giải như cảm xúc học tập. AFELO vẫn là proxy. Chương chưa quan sát và cặp chưa đủ điều kiện giữ nhãn null.


In [ ]:
import importlib.util
import subprocess
import sys

required = {'torch': 'torch==2.8.0', 'transformers': 'transformers==4.57.6',
            'huggingface_hub': 'huggingface-hub==0.36.2', 'duckdb': 'duckdb==1.3.2'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Python:', sys.version.split()[0], '| dependencies ready')


## Đường dẫn

Chạy sau notebook 06. Trên Kaggle, bảng nền phải ở thư mục có thể ghi `/kaggle/working/final_quality_tables`; bật Internet ở lần tải model đầu tiên. Khi đã có model trong cache, có thể đặt `ALLOW_DOWNLOAD=False`. Script và engine 06 đều được nhúng bên dưới.


In [ ]:
from pathlib import Path
import json
import os

BASE = Path.cwd() / 'working_13-10' if (Path.cwd() / 'working_13-10').is_dir() else Path.cwd()
if Path('/kaggle/working').is_dir():
    BASE = Path('/kaggle/working')
QUALITY_ROOT = BASE / 'final_quality_tables'
OUTPUT_ROOT = BASE / 'afelo_sentiment_cardiff'
CACHE_DIR = Path('/kaggle/working/hf-cache') if Path('/kaggle').is_dir() else Path('/private/tmp/paper3-hf-cache')
ENGINE_PATH = BASE / '06_build_final_quality_tables.py'
ALLOW_DOWNLOAD = True
DEVICE = 'auto'  # CUDA -> MPS -> CPU; default model FP32, no fine tuning
THREADS = 4
REUSE_COMPLETE = True
print('Quality:', QUALITY_ROOT, '\nSentiment:', OUTPUT_ROOT)


## Engine và công thức

Tiền xử lý giữ đúng model card: `@name → @user`, URL bắt đầu bằng `http → http`. Chỉ gộp văn bản giống hệt sau tiền xử lý khi suy luận, sau đó ánh xạ về tất cả comment. Không loại người dùng vì nội dung trùng.

`sentiment_valence = ((sum(p_pos) - sum(p_neg))/n_sentiment_comments + 1)/2`.

`AFELO_proxy = mean_non_null(comment_participation, reply_participation, response_received_rate, sentiment_valence)`.

Không có comment: sentiment/response rate null; vẫn dùng thành phần xã hội còn lại. Comment quá 512 token được cắt phần đầu và gắn cờ. Text có ký tự CJK được giữ và gắn cờ. Bộ lọc đầu vào loại rỗng/trùng chính xác, chưa xác định mọi comment ngoài nội dung học.


In [ ]:
FINALIZER_SOURCE = '"""Final-window user/chapter component tables, with AFELO deferred by default.\n\nUses notebook 01/02 inputs and notebook 05 time bounds. No phase features or\ntraining model is created. Missing chapters stay unobserved. The companion\nnotebook embeds this source and needs only DuckDB.\n"""\nfrom __future__ import annotations\n\nimport datetime as dt\nimport hashlib\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport duckdb\n\nFORMULA_VERSION = \'schema12_observed_chapters_v1\'\nPOLICY_VERSION = \'final-window-v1\'\nKEYS = \'user_id,course_id,lo_key\'\nMETA = \'user_id,course_id,chapter_major,lo_key,enroll_time,course_end,split\'\nCOE_INPUTS = [\'n_videos_available\',\'available_video_seconds_proxy\',\'n_videos_missing_duration\',\n    \'n_exercises_available\',\'n_problems_available\',\'n_videos_watched\',\'n_video_segments\',\n    \'n_invalid_video_segments\',\'raw_watch_seconds\',\'unique_watch_seconds\',\'rewatch_seconds\',\n    \'n_video_sessions_30m\',\'n_revisited_videos\',\'n_exercises_attempted\',\'n_problems_attempted\',\n    \'total_problem_attempts\',\'invalid_attempts\']\nAFE_INPUTS = [\'n_comments_raw\',\'n_comments_filtered\',\'n_comments_empty\',\'n_comments_duplicate\',\n    \'n_replies_authored\',\'n_user_comments_with_reply\']\nACE_INPUTS = [\'n_problems_available\',\'available_max_score_sum\',\'n_problems_missing_score\',\n    \'n_problems_attempted\',\'n_problems_correct\',\'n_first_try_correct\',\'total_problem_attempts\',\n    \'earned_score_sum\',\'attempted_max_score_sum\',\'efficiency_sum\',\'n_scores_valid\',\n    \'n_scores_fallback\',\'n_scores_unresolved\',\'n_efficiency_unresolved\',\'invalid_attempts\']\nCOUNT_FIELDS = [\'n_videos_watched\',\'n_video_segments\',\'n_invalid_video_segments\',\n    \'raw_watch_seconds\',\'unique_watch_seconds\',\'rewatch_seconds\',\'n_video_sessions_30m\',\n    \'n_revisited_videos\',\'n_exercises_attempted\',\'n_problems_attempted\',\'total_problem_attempts\',\n    \'n_problems_correct\',\'n_first_try_correct\',\'earned_score_sum\',\'attempted_max_score_sum\',\n    \'efficiency_sum\',\'n_scores_valid\',\'n_scores_fallback\',\'n_scores_unresolved\',\n    \'n_efficiency_unresolved\',\'invalid_attempts\',\'n_comments_raw\',\'n_comments_filtered\',\n    \'n_comments_empty\',\'n_comments_duplicate\',\'n_replies_authored\',\'n_user_comments_with_reply\']\n\n\ndef qs(value):\n    return "\'" + str(value).replace("\'", "\'\'") + "\'"\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef fingerprint(raw, model):\n    values = []\n    for kind, root in [(\'raw\',Path(raw)),(\'model\',Path(model))]:\n        for p in sorted(root.rglob(\'*.parquet\')):\n            s=p.stat(); values.append([kind,str(p.relative_to(root)),s.st_size,s.st_mtime_ns])\n    return hashlib.sha256(json.dumps(values,sort_keys=True).encode()).hexdigest()\n\n\ndef discover():\n    for parent in [Path.cwd(),Path.cwd()/\'working_13-10\']:\n        r=parent/\'input\'\n        if (r/\'moocubex_parquet/catalog.json\').exists() and (r/\'moocubex_model/catalog.json\').exists():\n            return r/\'moocubex_parquet\',r/\'moocubex_model\',parent/\'review_time_phases\'\n    found_raw,found_model,found_times=[],[],[]\n    for parent in [Path(\'/kaggle/input\'),Path(\'/kaggle/working\')]:\n        if parent.is_dir():\n            for p in parent.rglob(\'catalog.json\'):\n                if (p.parent/\'tables/relations/user_video\').is_dir(): found_raw.append(p.parent)\n                if (p.parent/\'processed/dim_lo_catalog\').is_dir(): found_model.append(p.parent)\n            for p in parent.rglob(\'user_course_phase_times.parquet\'):\n                found_times.append(p.parent.parent)\n    if len(found_raw)==len(found_model)==len(found_times)==1:\n        return found_raw[0],found_model[0],found_times[0]\n    raise RuntimeError(\'Set raw_root, model_root and time_root explicitly.\')\n\n\ndef open_db(out,memory_limit=\'2GB\',threads=2):\n    scratch=Path(out)/\'_scratch\';scratch.mkdir(parents=True,exist_ok=True)\n    con=duckdb.connect();con.execute(f\'SET threads={int(threads)}\')\n    con.execute(f\'SET memory_limit={qs(memory_limit)}\')\n    con.execute(f\'SET temp_directory={qs(scratch)}\')\n    con.execute("SET max_temp_directory_size=\'4GB\'")\n    con.execute("SET TimeZone=\'UTC\'")\n    con.execute(\'SET preserve_insertion_order=false\')\n    return con\n\n\ndef dictionaries(con,sql):\n    cursor=con.execute(sql);names=[c[0] for c in cursor.description]\n    return [dict(zip(names,row)) for row in cursor.fetchall()]\n\n\ndef fixture_checks():\n    con=duckdb.connect()\n    try:\n        # Retain a real partial score; use binary fallback only when score is invalid.\n        rows=con.execute("""WITH p(score,max_score,is_correct,attempts) AS (VALUES\n            (0.5,1.0,0,2),(NULL,2.0,1,2),(-1.0,2.0,0,3),(NULL,NULL,1,1))\n            SELECT CASE WHEN score>=0 AND score<=max_score AND max_score>0 THEN score\n                        WHEN max_score>0 THEN is_correct*max_score END earned,\n                   is_correct::DOUBLE/attempts efficiency FROM p""").fetchall()\n        assert rows==[(0.5,0.0),(2.0,0.5),(0.0,0.0),(None,1.0)]\n        assert con.execute(\'SELECT list_avg([NULL::DOUBLE,0.5]),list_count([NULL::DOUBLE,0.5])\').fetchone()==(0.5,1)\n        # Average distances after chapter-level calculation, not distance of averages.\n        q=(0.0+1.0)/2\n        vec_distance=math.sqrt(3*(1-0.5)**2)/math.sqrt(3)\n        assert abs(q-vec_distance)<1e-12\n        # Non-collinear chapter vectors show the aggregation difference.\n        q=(1.0/math.sqrt(3)+math.sqrt(2)/math.sqrt(3))/2\n        vec_distance=math.sqrt(3*(1-0.5)**2)/math.sqrt(3)\n        assert abs(q-vec_distance)>0.1\n        assert con.execute("SELECT count(*) FROM (VALUES (TIMESTAMP \'2020-05-01\'),(TIMESTAMP \'2020-05-03\'),(TIMESTAMP \'2020-05-05\')) t(event_time) WHERE event_time BETWEEN TIMESTAMP \'2020-05-02\' AND TIMESTAMP \'2020-05-04\'").fetchone()[0]==1\n        return {\'partial_score_preserved\':True,\'binary_fallback_only_when_needed\':True,\n                \'missing_max_score_stays_null\':True,\'mean_non_null_semantics\':True,\n                \'chapter_distance_before_aggregation\':True,\'final_window_filter\':True}\n    finally:\n        con.close()\n\n\ndef run_build(raw_root=None,model_root=None,time_root=None,output_root=None,memory_limit=\'2GB\',threads=2):\n    if raw_root is None or model_root is None or time_root is None:\n        a,b,c=discover();raw_root=raw_root or a;model_root=model_root or b;time_root=time_root or c\n    raw,model,times=map(lambda p:Path(p).resolve(),(raw_root,model_root,time_root))\n    out=Path(output_root or (\'/kaggle/working/final_quality_tables\' if Path(\'/kaggle\').is_dir()\n                            else raw.parent.parent/\'final_quality_tables\')).resolve()\n    for r in [raw,model,times]:\n        if r==out or r in out.parents or out in r.parents: raise ValueError(\'Output must be separate from inputs.\')\n    out.mkdir(parents=True,exist_ok=True)\n    tables=out/\'tables\';tables.mkdir(exist_ok=True)\n    stage=out/\'_stage\';stage.mkdir(exist_ok=True)\n    success=out/\'_SUCCESS.json\'\n    if success.exists():success.unlink()\n    con=open_db(out,memory_limit,threads)\n    base_fp=fingerprint(raw,model)\n    time_hash=sha256(times/\'run_manifest.json\')\n    run_signature=hashlib.sha256((base_fp+time_hash+POLICY_VERSION).encode()).hexdigest()\n    started=time.monotonic();log=[];lineage=[]\n    cache_path=out/\'step_checkpoints.json\'\n    cache=json.loads(cache_path.read_text()) if cache_path.exists() else {}\n\n    def view(name,path):\n        identifier=\'"\'+name.replace(\'"\',\'""\')+\'"\'\n        con.execute(f\'CREATE OR REPLACE VIEW {identifier} AS SELECT * FROM read_parquet({qs(path)},union_by_name=true)\')\n\n    def save(name,sql,folder=stage):\n        p=folder/f\'{name}.parquet\';tick=time.monotonic()\n        signature=hashlib.sha256((run_signature+\'\'.join(lineage)+sql).encode()).hexdigest()\n        reused=p.exists() and cache.get(str(p.relative_to(out)))==signature\n        if not reused:\n            partial=p.with_suffix(\'.parquet.partial\')\n            con.execute(f\'COPY ({sql}) TO {qs(partial)} (FORMAT PARQUET,COMPRESSION ZSTD,ROW_GROUP_SIZE 122880)\')\n            partial.replace(p);cache[str(p.relative_to(out))]=signature\n            cache_path.write_text(json.dumps(cache,indent=2))\n        view(name,p)\n        lineage.append(signature)\n        log.append({\'name\':name,\'sql\':sql,\'seconds\':round(time.monotonic()-tick,3),\'cached\':reused})\n        print(name,\'cached\' if reused else \'built\',log[-1][\'seconds\'],\'seconds\',flush=True)\n        return p\n\n    for name,relative in [(\'lo_catalog\',\'dim_lo_catalog\'),(\'resources\',\'dim_lo_resource\'),(\'lo_problem\',\'bridge_lo_problem\'),\n                           (\'course_comment\',\'bridge_course_comment\')]:\n        view(name,model/\'processed\'/relative/\'*.parquet\')\n    view(\'time_pairs\',times/\'tables/user_course_phase_times.parquet\')\n    view(\'course_bounds\',times/\'tables/course_time_bounds.parquet\')\n    for name,relative in [(\'uv\',\'relations/user_video\'),(\'up\',\'relations/user_problem\'),\n                           (\'comments\',\'entities/translated_comment\'),(\'replies\',\'entities/reply\'),(\'comment_reply\',\'relations/comment_reply\')]:\n        view(name,raw/\'tables\'/relative/\'*.parquet\')\n    con.execute(\'\'\'CREATE TEMP TABLE resource_map AS SELECT resource_id,min(course_id) course_id,min(lo_key) lo_key,\n        min(resource_type) resource_type FROM resources GROUP BY 1 HAVING count(DISTINCT lo_key)=1\'\'\')\n    con.execute(\'\'\'CREATE TEMP TABLE problem_map AS SELECT problem_id,min(course_id) course_id,min(lo_key) lo_key,\n        min(exercise_id) exercise_id,max(problem_max_score) problem_max_score,count(DISTINCT exercise_id) n_exercises\n        FROM lo_problem GROUP BY 1 HAVING count(DISTINCT lo_key)=1\'\'\')\n    con.execute(\'\'\'CREATE TEMP TABLE comment_course AS SELECT comment_id,min(course_id) course_id,\n        count(DISTINCT course_id) n_courses FROM course_comment GROUP BY 1\'\'\')\n    save(\'course_splits\', \'\'\'WITH eligible_courses AS (\n        SELECT DISTINCT course_id FROM time_pairs WHERE time_eligible\n      ), ranked AS (SELECT b.*,row_number() OVER(ORDER BY course_start,course_id) pos,\n          count(*) OVER() n_courses FROM course_bounds b JOIN eligible_courses e USING(course_id))\n      SELECT course_id,course_start,course_end,\n        CASE WHEN pos<=floor(n_courses*0.7) THEN \'train\'\n             WHEN pos<=floor(n_courses*0.8) THEN \'dev\' ELSE \'test\' END split\n      FROM ranked\'\'\',tables)\n    split_rows=dictionaries(con,\'SELECT course_id,split FROM course_splits ORDER BY course_id\')\n    (out/\'splits.json\').write_text(json.dumps({r[\'course_id\']:r[\'split\'] for r in split_rows},indent=2))\n\n    # Reuse the already validated mapped row stage, not its full-window aggregates.\n    prior_dir=raw.parent.parent/\'eda_engagement_features_comparison\'\n    prior_stage=prior_dir/\'_stage/problem_rows\'\n    prior_manifest=prior_dir/\'run_manifest.json\'\n    use_prior=prior_manifest.exists() and prior_stage.exists() and json.loads(prior_manifest.read_text()).get(\'input_fingerprint\')==base_fp\n    if use_prior:\n        problem_stage=prior_stage\n        print(\'Reuse verified mapped problem rows; re-filter each row by enrollment/end.\',flush=True)\n    else:\n        problem_stage=stage/\'problem_rows\';problem_stage.mkdir(exist_ok=True)\n        done=problem_stage/\'_SUCCESS.json\'\n        if not done.exists() or json.loads(done.read_text()).get(\'signature\')!=run_signature:\n            # Only our own stage is replaced on a changed input snapshot.\n            import shutil\n            shutil.rmtree(problem_stage);problem_stage.mkdir()\n            con.execute(f\'\'\'COPY (SELECT u.*,m.course_id,m.lo_key,m.exercise_id,m.problem_max_score,m.n_exercises,\n                try_cast(u.submit_time AS TIMESTAMP) event_time,\n                coalesce(isfinite(u.score) AND u.score>=0 AND m.problem_max_score>0 AND u.score<=m.problem_max_score+1e-9,false) score_valid,\n                coalesce(u.is_correct IN (0,1) AND m.problem_max_score>0 AND isfinite(m.problem_max_score),false) fallback_possible,\n                coalesce(u.attempts>0 AND u.is_correct IN (0,1),false) efficiency_valid,hash(u.user_id)%32 user_shard\n                FROM up u JOIN problem_map m USING(problem_id)) TO {qs(problem_stage)}\n                (FORMAT PARQUET,COMPRESSION ZSTD,PARTITION_BY(user_shard),ROW_GROUP_SIZE 50000)\'\'\')\n            done.write_text(json.dumps({\'signature\':run_signature}))\n    view(\'mapped_problems\',problem_stage/\'**/*.parquet\')\n\n    save(\'video_events\', \'\'\'WITH items AS (SELECT user_id,unnest(seq) item FROM uv),\n        segments AS (SELECT user_id,m.course_id,m.lo_key,item.video_id,unnest(item.segment) s\n          FROM items JOIN resource_map m ON item.video_id=m.resource_id AND m.resource_type=\'video\')\n      SELECT user_id,course_id,lo_key,video_id,s.start_point,s.end_point,s.speed,\n        cast(to_timestamp(s.local_start_time) AS TIMESTAMP) event_time,\n        coalesce(s.start_point>=0 AND s.end_point>s.start_point AND isfinite(s.start_point)\n          AND isfinite(s.end_point) AND s.speed>0 AND isfinite(s.speed),false) valid_measurement\n      FROM segments WHERE s.local_start_time>=epoch(TIMESTAMP \'2020-01-19\')\n        AND s.local_start_time<epoch(TIMESTAMP \'2020-11-04\')\'\'\')\n    save(\'comment_prepared\', \'\'\'WITH typed AS (\n        SELECT c.id comment_id,\'U_\'||cast(c.user_id AS VARCHAR) user_id,c.resource_id,c.text,\n          cast(to_timestamp(c.create_time/1000.0) AS TIMESTAMP) event_time,r.course_id,r.lo_key,\n          nullif(regexp_replace(trim(c.text),\'\\\\s+\',\' \',\'g\'),\'\') normalized_text\n        FROM comments c JOIN resource_map r ON c.resource_id=r.resource_id\n        LEFT JOIN comment_course cc ON c.id=cc.comment_id\n        WHERE c.create_time BETWEEN 0 AND 4102444800000 AND c.user_id IS NOT NULL\n          AND (cc.comment_id IS NULL OR (cc.n_courses=1 AND cc.course_id=r.course_id))\n      ), ids AS (SELECT *,row_number() OVER(PARTITION BY comment_id ORDER BY event_time,user_id,resource_id,text) id_rank FROM typed)\n      SELECT * EXCLUDE(id_rank),row_number() OVER(PARTITION BY user_id,resource_id,event_time,normalized_text ORDER BY comment_id) exact_duplicate_rank\n      FROM ids WHERE id_rank=1\'\'\')\n    con.execute(\'\'\'CREATE TEMP TABLE reply_parent AS SELECT reply_id,min(comment_id) parent_comment_id\n        FROM comment_reply GROUP BY 1 HAVING count(DISTINCT comment_id)=1\'\'\')\n    save(\'reply_prepared\', \'\'\'WITH typed AS (\n        SELECT r.id reply_id,r.user_id,try_cast(r.create_time AS TIMESTAMP) event_time,\n          p.parent_comment_id,c.event_time parent_time,c.course_id,c.lo_key,\n          row_number() OVER(PARTITION BY r.id ORDER BY r.create_time,r.user_id,r.text) id_rank\n        FROM replies r JOIN reply_parent p ON r.id=p.reply_id\n        JOIN comment_prepared c ON p.parent_comment_id=c.comment_id)\n      SELECT * EXCLUDE(id_rank) FROM typed WHERE id_rank=1 AND event_time>=parent_time\'\'\')\n    save(\'active_candidates\', f\'\'\'SELECT {KEYS.split(\',\')[0]},course_id FROM mapped_problems GROUP BY 1,2\n        UNION SELECT user_id,course_id FROM video_events\n        UNION SELECT user_id,course_id FROM comment_prepared\n        UNION SELECT user_id,course_id FROM reply_prepared\'\'\')\n    save(\'active_pairs\', \'\'\'SELECT t.user_id,t.course_id,t.enroll_time,t.course_end,s.split\n        FROM time_pairs t JOIN active_candidates a USING(user_id,course_id)\n        JOIN course_splits s USING(course_id) WHERE t.time_eligible\'\'\')\n    save(\'video_window\', \'\'\'SELECT v.* FROM video_events v JOIN active_pairs p USING(user_id,course_id)\n        WHERE v.event_time BETWEEN p.enroll_time AND p.course_end\'\'\')\n    save(\'video_union\', \'\'\'WITH intervals AS (\n        SELECT DISTINCT user_id,course_id,lo_key,video_id,start_point,end_point FROM video_window WHERE valid_measurement\n      ), previous AS (SELECT *,max(end_point) OVER(PARTITION BY user_id,course_id,lo_key,video_id\n          ORDER BY start_point,end_point ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) prev_end FROM intervals),\n      numbered AS (SELECT *,sum(CASE WHEN prev_end IS NULL OR start_point>prev_end THEN 1 ELSE 0 END)\n          OVER(PARTITION BY user_id,course_id,lo_key,video_id ORDER BY start_point,end_point) island FROM previous)\n      SELECT user_id,course_id,lo_key,video_id,island,min(start_point) interval_start,max(end_point) interval_end\n      FROM numbered GROUP BY 1,2,3,4,5\'\'\')\n    save(\'video_features\', \'\'\'WITH counts AS (\n        SELECT user_id,course_id,lo_key,count(*) FILTER(WHERE valid_measurement) n_video_segments,\n          count(*) FILTER(WHERE NOT valid_measurement) n_invalid_video_segments,\n          count(DISTINCT video_id) FILTER(WHERE valid_measurement) n_videos_watched,\n          coalesce(sum(end_point-start_point) FILTER(WHERE valid_measurement),0) raw_watch_seconds,\n          min(event_time) first_video_time,max(event_time) last_video_time FROM video_window GROUP BY 1,2,3\n      ), unique_seconds AS (SELECT user_id,course_id,lo_key,sum(interval_end-interval_start) unique_watch_seconds\n        FROM video_union GROUP BY 1,2,3), edges AS (\n        SELECT *,lag(event_time) OVER(PARTITION BY user_id,course_id,lo_key,video_id ORDER BY event_time,start_point,end_point) prev_time\n        FROM video_window WHERE valid_measurement), sessions AS (\n        SELECT user_id,course_id,lo_key,video_id,count(*) FILTER(WHERE prev_time IS NULL OR event_time-prev_time>INTERVAL \'30 minutes\') n_sessions\n        FROM edges GROUP BY 1,2,3,4), session_stats AS (\n        SELECT user_id,course_id,lo_key,sum(n_sessions) n_video_sessions_30m,count(*) FILTER(WHERE n_sessions>1) n_revisited_videos\n        FROM sessions GROUP BY 1,2,3)\n      SELECT c.*,coalesce(u.unique_watch_seconds,0) unique_watch_seconds,\n        c.raw_watch_seconds-coalesce(u.unique_watch_seconds,0) rewatch_seconds,\n        coalesce(s.n_video_sessions_30m,0) n_video_sessions_30m,coalesce(s.n_revisited_videos,0) n_revisited_videos\n      FROM counts c LEFT JOIN unique_seconds u USING(user_id,course_id,lo_key)\n      LEFT JOIN session_stats s USING(user_id,course_id,lo_key)\'\'\')\n\n    # Aggregate by user-hash shards to bound the distinct user/problem memory use.\n    problem_feature_dir=stage/\'problem_feature_shards\';problem_feature_dir.mkdir(exist_ok=True)\n    problem_agg=\'\'\'SELECT p.user_id,p.course_id,p.lo_key,count(*) n_problems_attempted,\n        count(DISTINCT p.exercise_id) n_exercises_attempted,sum(p.attempts) total_problem_attempts,\n        count(*) FILTER(WHERE p.attempts IS NULL OR p.attempts<=0) invalid_attempts,\n        count(*) FILTER(WHERE p.is_correct=1) n_problems_correct,\n        count(*) FILTER(WHERE p.is_correct=1 AND p.attempts=1) n_first_try_correct,\n        coalesce(sum(CASE WHEN p.score_valid THEN p.score WHEN p.fallback_possible THEN p.is_correct*p.problem_max_score END),0) earned_score_sum,\n        coalesce(sum(p.problem_max_score) FILTER(WHERE p.problem_max_score>0),0) attempted_max_score_sum,\n        coalesce(sum(CASE WHEN p.efficiency_valid THEN p.is_correct::DOUBLE/p.attempts END),0) efficiency_sum,\n        count(*) FILTER(WHERE p.score_valid) n_scores_valid,\n        count(*) FILTER(WHERE NOT p.score_valid AND p.fallback_possible) n_scores_fallback,\n        count(*) FILTER(WHERE NOT p.score_valid AND NOT p.fallback_possible) n_scores_unresolved,\n        count(*) FILTER(WHERE NOT p.efficiency_valid) n_efficiency_unresolved,\n        min(p.event_time) first_problem_time,max(p.event_time) last_problem_time\n        FROM {source} p JOIN active_pairs a USING(user_id,course_id)\n        WHERE p.event_time BETWEEN a.enroll_time AND a.course_end GROUP BY 1,2,3\'\'\'\n    for shard in range(32):\n        source=f\'read_parquet({qs(problem_stage/f"user_shard={shard}"/"*.parquet")})\'\n        save(f\'part-{shard:03d}\',problem_agg.format(source=source),problem_feature_dir)\n    view(\'problem_features\',problem_feature_dir/\'*.parquet\')\n    save(\'comment_window\', \'\'\'SELECT c.*,p.enroll_time,p.course_end FROM comment_prepared c\n        JOIN active_pairs p USING(user_id,course_id) WHERE c.event_time BETWEEN p.enroll_time AND p.course_end\'\'\')\n    save(\'afelo_comment_links\', \'\'\'SELECT * FROM comment_window\n        WHERE normalized_text IS NOT NULL AND exact_duplicate_rank=1\'\'\',tables)\n    save(\'social_features\', \'\'\'WITH received AS (\n        SELECT DISTINCT c.comment_id FROM afelo_comment_links c JOIN reply_prepared r\n          ON c.comment_id=r.parent_comment_id AND r.event_time>=c.event_time AND r.event_time<=c.course_end\n      ), counts AS (SELECT c.user_id,c.course_id,c.lo_key,count(*) n_comments_raw,\n          count(*) FILTER(WHERE c.normalized_text IS NOT NULL AND c.exact_duplicate_rank=1) n_comments_filtered,\n          count(*) FILTER(WHERE c.normalized_text IS NULL) n_comments_empty,\n          count(*) FILTER(WHERE c.exact_duplicate_rank>1) n_comments_duplicate,\n          count(*) FILTER(WHERE x.comment_id IS NOT NULL AND c.normalized_text IS NOT NULL AND c.exact_duplicate_rank=1) n_user_comments_with_reply,\n          min(c.event_time) first_comment_time,max(c.event_time) last_comment_time\n        FROM comment_window c LEFT JOIN received x USING(comment_id) GROUP BY 1,2,3),\n      reply_counts AS (SELECT r.user_id,r.course_id,r.lo_key,count(*) n_replies_authored,\n          min(r.event_time) first_reply_time,max(r.event_time) last_reply_time\n        FROM reply_prepared r JOIN active_pairs p USING(user_id,course_id)\n        WHERE r.event_time BETWEEN p.enroll_time AND p.course_end GROUP BY 1,2,3)\n      SELECT coalesce(c.user_id,r.user_id) user_id,coalesce(c.course_id,r.course_id) course_id,\n        coalesce(c.lo_key,r.lo_key) lo_key,c.* EXCLUDE(user_id,course_id,lo_key),\n        r.* EXCLUDE(user_id,course_id,lo_key) FROM counts c FULL JOIN reply_counts r USING(user_id,course_id,lo_key)\'\'\')\n    video_fields=[\'n_videos_watched\',\'n_video_segments\',\'n_invalid_video_segments\',\'raw_watch_seconds\',\n                  \'unique_watch_seconds\',\'rewatch_seconds\',\'n_video_sessions_30m\',\'n_revisited_videos\']\n    problem_fields=[x for x in COUNT_FIELDS if x not in video_fields+AFE_INPUTS]\n    select_counts=\',\'.join(f\'coalesce({alias}.{field},0) {field}\' for alias,fields in [(\'v\',video_fields),(\'p\',problem_fields),(\'s\',AFE_INPUTS)] for field in fields)\n    save(\'chapter_inputs\',f\'\'\'WITH keys AS (\n        SELECT {KEYS} FROM video_features UNION SELECT {KEYS} FROM problem_features UNION SELECT {KEYS} FROM social_features)\n      SELECT k.*,d.* EXCLUDE(course_id,lo_key),a.enroll_time,a.course_end,a.split,\n        v.user_id IS NOT NULL has_video,p.user_id IS NOT NULL has_problem,s.user_id IS NOT NULL has_social,\n        {select_counts},least(v.first_video_time,p.first_problem_time,s.first_comment_time,s.first_reply_time) first_observed_event,\n        greatest(v.last_video_time,p.last_problem_time,s.last_comment_time,s.last_reply_time) last_observed_event\n      FROM keys k JOIN lo_catalog d USING(course_id,lo_key) JOIN active_pairs a USING(user_id,course_id)\n      LEFT JOIN video_features v USING(user_id,course_id,lo_key) LEFT JOIN problem_features p USING(user_id,course_id,lo_key)\n      LEFT JOIN social_features s USING(user_id,course_id,lo_key)\'\'\')\n    save(\'chapter_numeric\', \'\'\'SELECT *,\n        unique_watch_seconds/nullif(available_video_seconds_proxy,0) TS_partial_catalog,\n        CASE WHEN n_videos_missing_duration=0 AND available_video_seconds_proxy>0\n          THEN least(1.0,greatest(0.0,unique_watch_seconds/available_video_seconds_proxy)) END TS,\n        n_videos_watched::DOUBLE/nullif(n_videos_available,0) RV,\n        n_exercises_attempted::DOUBLE/nullif(n_exercises_available,0) AV,\n        CASE WHEN invalid_attempts=0 THEN total_problem_attempts::DOUBLE/nullif(n_problems_available,0) END IF_raw,\n        CASE WHEN n_problems_missing_score=0 AND n_scores_unresolved=0 THEN earned_score_sum/nullif(available_max_score_sum,0) END KL,\n        CASE WHEN n_efficiency_unresolved=0 THEN efficiency_sum/nullif(n_problems_available,0) END CFA,\n        earned_score_sum/nullif(attempted_max_score_sum,0) mastery_attempted_partial,\n        n_problems_attempted::DOUBLE/nullif(n_problems_available,0) problem_participation\n      FROM chapter_inputs\'\'\')\n    p95=con.execute("SELECT quantile_cont(IF_raw,0.95) FROM chapter_numeric WHERE split=\'train\' AND IF_raw IS NOT NULL").fetchone()[0]\n    if p95 is None or not math.isfinite(p95) or p95<=0:\n        raise RuntimeError(\'Train-only P95_IF_raw must be finite and positive; inspect training population.\')\n    constants={\'formula_version\':FORMULA_VERSION,\'P95_IF_raw_training\':p95,\n        \'reference\':\'one final-window row per observed user-course-chapter in train only\',\n        \'coelo_min_components\':2,\'acelo_min_components\':1,\'unobserved_chapters\':\'exclude; do not assign LOQV=1\',\n        \'q_requires_all_observed_chapters_scored\':True,\'split_method\':\'70/10/20 courses ordered by observed course_start, then course_id\'}\n    (out/\'protocol_constants.json\').write_text(json.dumps(constants,ensure_ascii=False,indent=2))\n    save(\'coelo_components\',f\'\'\'WITH normalized AS (\n        SELECT *,CASE WHEN IF_raw IS NOT NULL THEN least(1.0,greatest(0.0,ln(1+IF_raw)/ln(1+{repr(p95)}))) END IF_value FROM chapter_numeric),\n      counted AS (SELECT *,list_count([TS,RV,AV,IF_value]) coelo_n_components FROM normalized)\n      SELECT {META},{\',\'.join(COE_INPUTS)},TS_partial_catalog,TS,RV,AV,IF_raw,IF_value AS "IF",\n        coelo_n_components,CASE WHEN coelo_n_components>=2 THEN list_avg([TS,RV,AV,IF_value]) END COELO,\n        \'caption_proxy\' duration_source,{qs(FORMULA_VERSION)} formula_version FROM counted\'\'\',tables)\n    save(\'acelo_components\',f\'\'\'SELECT {META},{\',\'.join(ACE_INPUTS)},KL,CFA,\n        list_count([KL,CFA]) acelo_n_components,list_avg([KL,CFA]) ACELO,\n        mastery_attempted_partial,problem_participation,\n        CASE WHEN KL IS NOT NULL AND CFA IS NOT NULL THEN \'KL_AND_CFA\'\n             WHEN CFA IS NOT NULL THEN \'CFA_ONLY\' WHEN KL IS NOT NULL THEN \'KL_ONLY\' ELSE \'UNAVAILABLE\' END acelo_status,\n        {qs(FORMULA_VERSION)} formula_version FROM chapter_numeric\'\'\',tables)\n    # Base provenance is written before finalization, so AFELO can be filled later.\n    checks={\'fixtures\':fixture_checks(),\n        \'chapter_inputs_unique\':con.execute(f\'SELECT count(*)=count(DISTINCT ({KEYS})) FROM chapter_inputs\').fetchone()[0],\n        \'chapter_events_inside_final_window\':con.execute(\'SELECT count(*)=0 FROM chapter_inputs WHERE first_observed_event<enroll_time OR last_observed_event>course_end\').fetchone()[0],\n        \'source_tables_same_count\':con.execute(\'SELECT (SELECT count(*) FROM chapter_inputs)=(SELECT count(*) FROM coelo_components) AND (SELECT count(*) FROM chapter_inputs)=(SELECT count(*) FROM acelo_components)\').fetchone()[0],\n        \'all_observed_chapter_keys_preserved\':con.execute(f\'\'\'SELECT count(*)=0 FROM (\n            SELECT {KEYS} FROM video_features UNION SELECT {KEYS} FROM problem_features\n            UNION SELECT {KEYS} FROM social_features) k ANTI JOIN chapter_inputs b USING(user_id,course_id,lo_key)\'\'\').fetchone()[0],\n        \'video_union_not_greater_than_raw\':con.execute(\'SELECT count(*)=0 FROM chapter_inputs WHERE unique_watch_seconds>raw_watch_seconds+1e-8\').fetchone()[0],\n        \'received_reply_bounded\':con.execute(\'SELECT count(*)=0 FROM chapter_inputs WHERE n_user_comments_with_reply>n_comments_filtered\').fetchone()[0],\n        \'component_values_bounded\':con.execute(\'SELECT count(*)=0 FROM chapter_numeric WHERE RV>1+1e-9 OR AV>1+1e-9 OR KL>1+1e-9 OR CFA>1+1e-9 OR KL<0 OR CFA<0\').fetchone()[0],\n    }\n    assert all(v for k,v in checks.items() if k!=\'fixtures\'),checks\n    manifest={\'created_at_utc\':dt.datetime.now(dt.timezone.utc).isoformat(),\'duckdb_version\':duckdb.__version__,\n        \'raw_root\':str(raw),\'model_root\':str(model),\'time_root\':str(times),\'input_fingerprint\':base_fp,\n        \'time_manifest_sha256\':time_hash,\'reused_validated_problem_stage\':use_prior,\'sampling\':\'none\',\n        \'elapsed_seconds\':round(time.monotonic()-started,3),\'queries\':log,\'base_checks\':checks,\n        \'policies\':{\'window\':\'inclusive enroll_time to course_end; all sources filtered before aggregation\',\n          \'missing_chapters\':\'not materialized; unobserved, not LOQV=1\',\n          \'video\':\'valid timestamp in paper window; invalid intervals counted, excluded from watch metrics; union intervals\',\n          \'problem\':\'user-problem aggregate assigned at submit_time; no invented individual attempt times\',\n          \'social_filter\':\'remove empty text and exact same user/resource/time/normalized-text duplicates; no short-text length threshold\',\n          \'labels\':\'AFELO deferred by user; no fabricated q/y; all observed chapters must be scorable before final q\',\n          \'roles\':\'final-window label/audit tables only; not P1-P4 model features\'}}\n    (out/\'run_manifest.json\').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str))\n    con.close()\n    return finalize_quality(out,memory_limit=memory_limit,threads=threads)\n\n\ndef finalize_quality(output_root,sentiment_path=None,afelo_mode=\'deferred\',memory_limit=\'2GB\',threads=2):\n    """Fill AFELO and labels from persisted base tables, without raw data scans.\n\n    sentiment input parquet: comment_id,p_pos,p_neu,p_neg. The optional\n    social_only mode is explicit; default deferred keeps AFELO/LOQV/q/y null.\n    """\n    if afelo_mode not in [\'deferred\',\'sentiment\',\'social_only\']:\n        raise ValueError(\'afelo_mode must be deferred, sentiment or social_only\')\n    if afelo_mode==\'sentiment\' and sentiment_path is None:\n        raise ValueError(\'Provide sentiment_path with comment_id,p_pos,p_neu,p_neg.\')\n    out=Path(output_root).resolve();tables=out/\'tables\';stage=out/\'_stage\'\n    con=open_db(out,memory_limit,threads)\n    manifest=json.loads((out/\'run_manifest.json\').read_text());times=Path(manifest[\'time_root\'])\n    def view(name,path):\n        identifier=\'"\'+name.replace(\'"\',\'""\')+\'"\'\n        con.execute(f\'CREATE OR REPLACE VIEW {identifier} AS SELECT * FROM read_parquet({qs(path)})\')\n    def save(name,sql,folder=tables):\n        p=folder/f\'{name}.parquet\';partial=p.with_suffix(\'.parquet.partial\')\n        con.execute(f\'COPY ({sql}) TO {qs(partial)} (FORMAT PARQUET,COMPRESSION ZSTD,ROW_GROUP_SIZE 122880)\')\n        partial.replace(p);view(name,p);print(name,\'built\',flush=True)\n        return p\n    for name in [\'chapter_numeric\',\'chapter_inputs\']:view(name,stage/f\'{name}.parquet\')\n    for name in [\'coelo_components\',\'acelo_components\',\'afelo_comment_links\',\'course_splits\']:view(name,tables/f\'{name}.parquet\')\n    view(\'time_pairs\',times/\'tables/user_course_phase_times.parquet\')\n    view(\'lo_catalog\',Path(manifest[\'model_root\'])/\'processed/dim_lo_catalog/*.parquet\')\n    if afelo_mode==\'sentiment\':\n        view(\'sentiment\',Path(sentiment_path).resolve())\n        invalid=con.execute(\'\'\'SELECT count(*) FROM sentiment WHERE comment_id IS NULL OR p_pos IS NULL OR p_neu IS NULL OR p_neg IS NULL\n            OR NOT isfinite(p_pos) OR NOT isfinite(p_neu) OR NOT isfinite(p_neg)\n            OR p_pos<0 OR p_neu<0 OR p_neg<0 OR p_pos>1 OR p_neu>1 OR p_neg>1 OR abs(p_pos+p_neu+p_neg-1)>1e-6\'\'\').fetchone()[0]\n        assert invalid==0,\'Invalid sentiment probability rows\'\n        assert con.execute(\'SELECT count(*)=count(DISTINCT comment_id) FROM sentiment\').fetchone()[0],\'Duplicate sentiment comment IDs\'\n        con.execute(\'\'\'CREATE TEMP VIEW sentiment_summary AS SELECT c.user_id,c.course_id,c.lo_key,\n            count(*) n_sentiment_comments,sum(s.p_pos) sentiment_pos_sum,sum(s.p_neu) sentiment_neu_sum,sum(s.p_neg) sentiment_neg_sum\n            FROM afelo_comment_links c JOIN sentiment s USING(comment_id) GROUP BY 1,2,3\'\'\')\n    else:\n        con.execute(\'\'\'CREATE TEMP VIEW sentiment_summary AS SELECT NULL::VARCHAR user_id,NULL::VARCHAR course_id,NULL::VARCHAR lo_key,\n            0::BIGINT n_sentiment_comments,NULL::DOUBLE sentiment_pos_sum,NULL::DOUBLE sentiment_neu_sum,NULL::DOUBLE sentiment_neg_sum WHERE false\'\'\')\n    success=out/\'_SUCCESS.json\'\n    if success.exists():success.unlink()\n    score_expr=\'NULL::DOUBLE\' if afelo_mode==\'deferred\' else \'list_avg([comment_participation,reply_participation,response_received_rate,sentiment_valence])\'\n    save(\'afelo_components\',f\'\'\'WITH inputs AS (\n        SELECT b.*,coalesce(s.n_sentiment_comments,0) n_sentiment_comments,\n          s.sentiment_pos_sum,s.sentiment_neu_sum,s.sentiment_neg_sum,\n          (b.n_comments_filtered>0)::DOUBLE comment_participation,(b.n_replies_authored>0)::DOUBLE reply_participation,\n          b.n_user_comments_with_reply::DOUBLE/nullif(b.n_comments_filtered,0) response_received_rate,\n          CASE WHEN s.n_sentiment_comments>0 THEN ((s.sentiment_pos_sum-s.sentiment_neg_sum)/s.n_sentiment_comments+1)/2 END sentiment_valence\n        FROM chapter_numeric b LEFT JOIN sentiment_summary s USING(user_id,course_id,lo_key))\n      SELECT {META},{\',\'.join(AFE_INPUTS)},n_sentiment_comments,sentiment_pos_sum,sentiment_neu_sum,sentiment_neg_sum,\n        comment_participation,reply_participation,response_received_rate,sentiment_valence,\n        n_sentiment_comments::DOUBLE/nullif(n_comments_raw,0) afe_text_coverage,\n        list_avg([comment_participation,reply_participation,response_received_rate]) AFELO_social_proxy,\n        {score_expr} AFELO_proxy,\n        list_count([comment_participation,reply_participation,response_received_rate,sentiment_valence]) afelo_n_components,\n        {qs(afelo_mode)} afelo_status,{qs(FORMULA_VERSION)} formula_version FROM inputs\'\'\')\n    save(\'chapter_quality\',f\'\'\'WITH joined AS (\n        SELECT b.*,c.TS_partial_catalog,c.TS,c.RV,c.AV,c.IF_raw,c."IF",c.COElO,c.coelo_n_components,\n          a.comment_participation,a.reply_participation,a.response_received_rate,a.sentiment_valence,\n          a.n_sentiment_comments,a.sentiment_pos_sum,a.sentiment_neu_sum,a.sentiment_neg_sum,a.afe_text_coverage,\n          a.AFELO_social_proxy,a.AFELO_proxy,a.afelo_n_components,a.afelo_status,\n          x.KL,x.CFA,x.mastery_attempted_partial,x.problem_participation,x.acelo_n_components,x.acelo_status,x.ACELO,\n          c.user_id IS NOT NULL coelo_row_found,a.user_id IS NOT NULL afelo_row_found,x.user_id IS NOT NULL acelo_row_found\n        FROM chapter_inputs b LEFT JOIN coelo_components c USING(user_id,course_id,lo_key)\n        LEFT JOIN afelo_components a USING(user_id,course_id,lo_key) LEFT JOIN acelo_components x USING(user_id,course_id,lo_key))\n      SELECT *,CASE WHEN COELO IS NOT NULL AND AFELO_proxy IS NOT NULL AND ACELO IS NOT NULL\n        THEN sqrt(pow(1-COELO,2)+pow(1-AFELO_proxy,2)+pow(1-ACELO,2))/sqrt(3.0) END LOQV,\n        CASE WHEN NOT coelo_row_found OR NOT afelo_row_found OR NOT acelo_row_found THEN \'MISSING_COMPONENT_ROW\'\n             WHEN {qs(afelo_mode)}=\'deferred\' THEN \'PENDING_AFELO\'\n             WHEN COELO IS NULL OR AFELO_proxy IS NULL OR ACELO IS NULL THEN \'INCOMPLETE_COMPONENTS\' ELSE \'SCORABLE\' END chapter_status,\n        {qs(FORMULA_VERSION)} formula_version FROM joined\'\'\')\n    # Keep a nested list of all per-chapter fields as well as scalar aggregates.\n    summands=\',\'.join(f\'sum({name}) {name}_sum_observed\' for name in COUNT_FIELDS)\n    diagnostic_means=\',\'.join(f\'avg({name}) {name}_mean_chapter\' for name in [\'TS\',\'RV\',\'AV\',\'IF_raw\',\'"IF"\',\'KL\',\'CFA\',\'mastery_attempted_partial\',\'problem_participation\']\n        ).replace(\'"IF"_mean_chapter\',\'IF_mean_chapter\')\n    aggregate_sql=f\'\'\'SELECT user_id,course_id,count(*) n_chapters_observed,\n        count(LOQV) n_chapters_scored,avg(COELO) coelo_mean_chapter,avg(AFELO_proxy) afelo_proxy_mean_chapter,\n        avg(ACELO) acelo_mean_chapter,{diagnostic_means},{summands},\n        CASE WHEN count(LOQV)=count(*) THEN avg(LOQV) END q,\n        list(chapter_quality ORDER BY chapter_major) chapter_details\n      FROM chapter_quality WHERE hash(user_id)%32={{shard}} GROUP BY user_id,course_id\'\'\'\n    aggregate_dir=stage/\'user_course_aggregates\';aggregate_dir.mkdir(exist_ok=True)\n    final_dir=tables/\'user_course_final\';final_dir.mkdir(exist_ok=True)\n    # Nested per-chapter structs cannot spill efficiently in one global list()\n    # aggregate. Bound both the aggregation and the final join by user shards.\n    pair_dir=stage/\'eligible_pair_shards\';pair_done=pair_dir/\'_SUCCESS.json\'\n    if not pair_done.exists() or json.loads(pair_done.read_text()).get(\'time_sha256\')!=manifest[\'time_manifest_sha256\']:\n        import shutil\n        if pair_dir.exists():shutil.rmtree(pair_dir)\n        pair_dir.mkdir()\n        con.execute(\'SET partitioned_write_max_open_files=8\')\n        con.execute(f\'\'\'COPY (SELECT user_id,course_id,enroll_time,course_end,hash(user_id)%32 user_shard\n            FROM time_pairs WHERE time_eligible) TO {qs(pair_dir)}\n            (FORMAT PARQUET,COMPRESSION ZSTD,PARTITION_BY(user_shard),ROW_GROUP_SIZE 50000)\'\'\')\n        pair_done.write_text(json.dumps({\'time_sha256\':manifest[\'time_manifest_sha256\']}))\n    final_sql=f\'\'\'WITH chapter_counts AS (SELECT course_id,count(*) n_chapters_catalog FROM lo_catalog GROUP BY 1),\n      joined AS (SELECT t.user_id,t.course_id,t.enroll_time,t.course_end,s.split,c.n_chapters_catalog,\n        coalesce(a.n_chapters_observed,0) n_chapters_observed,coalesce(a.n_chapters_scored,0) n_chapters_scored,\n        c.n_chapters_catalog-coalesce(a.n_chapters_observed,0) n_chapters_unobserved,\n        a.* EXCLUDE(user_id,course_id,n_chapters_observed,n_chapters_scored)\n        FROM pair_slice t LEFT JOIN course_splits s USING(course_id) LEFT JOIN chapter_counts c USING(course_id)\n        LEFT JOIN aggregate_slice a USING(user_id,course_id))\n      SELECT *,CASE WHEN q<0.2 THEN \'E\' WHEN q<0.4 THEN \'G\' WHEN q>=0.4 THEN \'A\' END y,\n        CASE WHEN n_chapters_observed=0 THEN \'NO_OBSERVED_CHAPTER\'\n             WHEN {qs(afelo_mode)}=\'deferred\' THEN \'PENDING_AFELO\'\n             WHEN q IS NULL THEN \'INCOMPLETE_OBSERVED_CHAPTERS\' ELSE \'LABELED_OBSERVED_CHAPTERS\' END label_status,\n        {qs(FORMULA_VERSION)} formula_version FROM joined\'\'\'\n    for shard in range(32):\n        aggregate_path=save(f\'part-{shard:03d}\',aggregate_sql.format(shard=shard),aggregate_dir)\n        view(\'aggregate_slice\',aggregate_path)\n        pair_path=pair_dir/f\'user_shard={shard}\'\n        if list(pair_path.glob(\'*.parquet\')):\n            view(\'pair_slice\',pair_path/\'*.parquet\')\n        else:\n            con.execute(\'CREATE OR REPLACE VIEW pair_slice AS SELECT user_id,course_id,enroll_time,course_end FROM time_pairs WHERE false\')\n        save(f\'part-{shard:03d}\',final_sql,final_dir)\n    view(\'user_course_aggregates\',aggregate_dir/\'*.parquet\')\n    view(\'user_course_final\',final_dir/\'*.parquet\')\n    summary={\n        \'chapter_rows\':con.execute(\'SELECT count(*) FROM chapter_quality\').fetchone()[0],\n        \'user_course_rows\':con.execute(\'SELECT count(*) FROM user_course_final\').fetchone()[0],\n        \'afelo_mode\':afelo_mode,\n        \'chapter_status\':dictionaries(con,\'SELECT chapter_status,count(*) n_rows FROM chapter_quality GROUP BY 1\'),\n        \'label_status\':dictionaries(con,\'SELECT label_status,count(*) n_rows FROM user_course_final GROUP BY 1 ORDER BY n_rows DESC\'),\n        \'acelo_status\':dictionaries(con,\'SELECT acelo_status,count(*) n_rows FROM acelo_components GROUP BY 1\'),\n        \'split_summary\':dictionaries(con,\'SELECT split,count(*) n_courses,min(course_start) first_start,max(course_start) last_start FROM course_splits GROUP BY 1 ORDER BY first_start\'),\n        \'class_distribution\':dictionaries(con,\'SELECT split,y,count(*) n_rows FROM user_course_final WHERE y IS NOT NULL GROUP BY 1,2 ORDER BY 1,2\'),\n    }\n    overlap=dictionaries(con,"""SELECT count(*) n_train_courses,count(*) FILTER(WHERE course_end>(SELECT min(course_start) FROM course_splits WHERE split=\'test\')) n_train_ending_after_first_test_start\n        FROM course_splits WHERE split=\'train\'""")[0]\n    summary[\'temporal_overlap\']=overlap\n    checks={\n        \'chapter_tables_unique\':all(con.execute(f\'SELECT count(*)=count(DISTINCT ({KEYS})) FROM {name}\').fetchone()[0] for name in [\'coelo_components\',\'afelo_components\',\'acelo_components\',\'chapter_quality\']),\n        \'chapter_joins_preserve_count\':con.execute(\'SELECT count(*)=(SELECT count(*) FROM chapter_numeric) FROM chapter_quality\').fetchone()[0],\n        \'final_user_course_unique\':con.execute(\'SELECT count(*)=count(DISTINCT (user_id,course_id)) FROM user_course_final\').fetchone()[0],\n        \'all_time_eligible_pairs_preserved\':summary[\'user_course_rows\']==con.execute(\'SELECT count(*) FROM time_pairs WHERE time_eligible\').fetchone()[0],\n        \'final_rows_have_split_metadata\':con.execute(\'SELECT count(*)=0 FROM user_course_final WHERE split IS NULL\').fetchone()[0],\n        \'unobserved_users_not_assigned_zero_or_label\':con.execute(\'SELECT count(*)=0 FROM user_course_final WHERE n_chapters_observed=0 AND (q IS NOT NULL OR y IS NOT NULL OR coelo_mean_chapter IS NOT NULL OR acelo_mean_chapter IS NOT NULL)\').fetchone()[0],\n        \'nested_details_match_observed_chapter_count\':con.execute(\'SELECT count(*)=0 FROM user_course_final WHERE coalesce(len(chapter_details),0)<>n_chapters_observed\').fetchone()[0],\n        \'label_thresholds_correct\':con.execute("SELECT count(*)=0 FROM user_course_final WHERE y IS DISTINCT FROM CASE WHEN q<0.2 THEN \'E\' WHEN q<0.4 THEN \'G\' WHEN q>=0.4 THEN \'A\' END").fetchone()[0],\n        \'no_label_when_observed_chapter_incomplete\':con.execute(\'SELECT count(*)=0 FROM user_course_final WHERE n_chapters_scored<n_chapters_observed AND q IS NOT NULL\').fetchone()[0],\n    }\n    if afelo_mode==\'deferred\':\n        checks[\'deferred_afelo_keeps_loqv_q_y_null\']=con.execute(\'SELECT (SELECT count(*) FROM chapter_quality WHERE AFELO_proxy IS NOT NULL OR LOQV IS NOT NULL)=0 AND (SELECT count(*) FROM user_course_final WHERE q IS NOT NULL OR y IS NOT NULL)=0\').fetchone()[0]\n    for table in [\'coelo_components\',\'afelo_components\',\'acelo_components\']:\n        checks[f\'{table}_has_all_observed_keys\']=con.execute(f\'\'\'SELECT count(*)=0\n            FROM chapter_inputs b ANTI JOIN {table} c USING(user_id,course_id,lo_key)\'\'\').fetchone()[0]\n    assert all(checks.values()),checks\n    summary[\'validation\']=checks\n    manifest[\'finalization\']={\'afelo_mode\':afelo_mode,\'sentiment_path\':str(sentiment_path) if sentiment_path else None,\n        \'sentiment_sha256\':sha256(sentiment_path) if sentiment_path else None,\'checks\':checks}\n    manifest[\'engine_source_sha256\']=globals().get(\'ENGINE_SOURCE_SHA256\') or sha256(__file__)\n    (out/\'run_manifest.json\').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str))\n    (out/\'summary.json\').write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str))\n    (out/\'validation.json\').write_text(json.dumps({\'base_checks\':manifest[\'base_checks\'],\'final_checks\':checks},ensure_ascii=False,indent=2))\n    (out/\'_SUCCESS.json\').write_text(json.dumps({\'afelo_mode\':afelo_mode,\'manifest_sha256\':sha256(out/\'run_manifest.json\')},indent=2))\n    write_report(out)\n    con.close();print(\'COMPLETE\',json.dumps(summary,ensure_ascii=False,default=str),flush=True)\n    return summary\n\n\ndef write_report(out):\n    out=Path(out);s=json.loads((out/\'summary.json\').read_text());c=json.loads((out/\'protocol_constants.json\').read_text())\n    fmt=lambda n:f\'{int(n):,}\'.replace(\',\',\'.\')\n    pending=s[\'afelo_mode\']==\'deferred\'\n    intro=(\'Bản hiện tại hoàn thành COELO và ACELO theo schema mục 12. AFELO được hoãn theo yêu cầu; \'\n        \'đã tính thành phần xã hội, giữ sentiment và AFELO hoàn chỉnh là null. Vì vậy LOQV, q và y đang chờ AFELO.\'\n        if pending else f\'AFELO đã được hoàn thiện ở mode {s["afelo_mode"]}; LOQV và nhãn chỉ tính khi đủ điều kiện coverage.\')\n    lines=[\'# Các bảng chất lượng cuối kỳ theo user–course–chapter\',\'\',intro,\'\',\n        f\'- Số user–course–chapter quan sát được: {fmt(s["chapter_rows"])}.\',\n        f\'- Số user–course ở bảng cuối: {fmt(s["user_course_rows"])}.\',\n        f\'- P95_IF_raw fit trên train: {c["P95_IF_raw_training"]:.12g}.\',\'\',\n        \'| Bảng | Grain | Nội dung |\',\'|---|---|---|\',\n        \'| coelo_components.parquet | user–course–chapter | Counts, thời lượng, mẫu số; TS, RV, AV, IF; COELO |\',\n        \'| afelo_components.parquet | user–course–chapter | Comment/reply, social proxy, sentiment, AFELO và trạng thái |\',\n        \'| acelo_components.parquet | user–course–chapter | Điểm, attempts, provenance fallback; KL, CFA, ACELO |\',\n        \'| chapter_quality.parquet | user–course–chapter | Toàn bộ feature con và chỉ số, LOQV và trạng thái |\',\n        \'| user_course_final/*.parquet | user–course | Một bảng chia 32 file; feature tổng hợp, ba chỉ số trung bình, q/y và chapter_details |\',\'\',\n        \'Các file nằm trong `tables/`. Mọi user–course đủ điều kiện thời gian từ notebook 05 đều được giữ trong bảng cuối. \'\n        \'Người không có chương quan sát được có chỉ số null và trạng thái NO_OBSERVED_CHAPTER.\',\'\',\n        \'Bảng chapter_quality lấy chapter_inputs làm bảng gốc, LEFT JOIN ba bảng thành phần theo user–course–lo_key. \'\n        \'Bảng cuối lấy mọi cặp đủ điều kiện thời gian làm gốc rồi LEFT JOIN kết quả tổng hợp. \'\n        \'Giá trị chỉ số null không làm mất dòng; các cờ *_row_found phân biệt thiếu bản ghi với có bản ghi nhưng thiếu chỉ số.\',\'\',\n        \'## Quy tắc tính\',\'\',\n        \'- Cửa sổ mỗi người: `[enroll_time, course_end]`. Mọi event được lọc trước khi aggregate.\',\n        \'- Chương không có log không tạo dòng và không tự gán LOQV=1; đây là lựa chọn người dùng.\',\n        \'- COELO = mean_non_null(TS, RV, AV, IF), tối thiểu 2 thành phần. TS dùng caption proxy đầy đủ; \'\n        \'giữ TS_partial_catalog để kiểm toán. IF chuẩn hóa bằng P95 của các dòng user–chapter cuối kỳ train.\',\n        \'- ACELO = mean_non_null(KL, CFA). KL chỉ có khi đủ mẫu số điểm của toàn chương và không còn score chưa giải được. \'\n        \'CFA_ONLY được gắn cờ riêng, không che phần KL thiếu.\',\n        \'- Score hợp lệ được giữ, kể cả điểm từng phần. Chỉ dùng is_correct × max_score khi score không hợp lệ và max_score hợp lệ.\',\n        (\'- AFELO_social_proxy là diagnostic riêng. AFELO_proxy đang chờ sentiment ở mode deferred.\' if pending else\n         \'- AFELO_proxy = mean_non_null(comment_participation, reply_participation, response_received_rate, sentiment_valence). \'\n         \'Chương không có comment giữ sentiment/response rate null; vẫn dùng các thành phần tham gia xã hội còn lại.\'),\n        \'- Khi hoàn thiện AFELO: LOQV tính từng chương; q chỉ lấy trung bình khi mọi chương quan sát của người đó đều có LOQV. \'\n        \'Không tính LOQV từ vector ba chỉ số đã trung bình.\',\n        \'- chapter_details là danh sách struct, giữ nguyên dữ liệu từng chương. Các feature scalar cộng có hậu tố sum_observed, \'\n        \'tỷ lệ/chỉ số trung bình chương có hậu tố mean_chapter; không cộng tỷ lệ.\',\n        \'- Không tự đặt n_problems_available=1 hoặc điểm tối đa=1 khi thiếu catalog.\',\n        \'\', \'## Trạng thái bảng cuối\',\'\', \'| Trạng thái | Số dòng |\',\'|---|---:|\']\n    lines.extend(f\'| {r["label_status"]} | {fmt(r["n_rows"])} |\' for r in s[\'label_status\'])\n    lines += [\'\', \'## Tách train/dev/test và giới hạn thời gian\',\'\',\n        \'Khóa được xếp theo observed course_start rồi course_id, chia 70/10/20 trước khi fit P95. \'\n        \'Không nhóm theo tên; course_id là đơn vị chia theo lựa chọn người dùng. Split hiện là bản sơ bộ để dựng chỉ số, \'\n        \'cần rà lại khi hoàn thiện nhãn và đánh giá mô hình.\',\n        f\'Train có {s["temporal_overlap"]["n_train_courses"]} khóa; \'\n        f\'{s["temporal_overlap"]["n_train_ending_after_first_test_start"]} khóa kết thúc sau ngày mở đầu tiên của test. \'\n        \'Theo review, cần kiểm tra thêm phương án loại các khóa chồng thời gian nếu chạy thí nghiệm vận hành.\',\n        \'Đây là bảng cuối kỳ phục vụ nhãn/kiểm toán, không phải X của P1–P4. Không đưa chỉ số nhãn, q/y, cờ thiếu \'\n        \'hoặc thông tin cuối kỳ vào X. Mốc course_end suy ra từ log là mốc hồi cứu.\',\n        \'`user_problem` là trạng thái tổng hợp tại submit_time; không khôi phục được timestamp từng attempt. \'\n        \'Nguồn video dùng khoảng thu thập công bố; các nguồn có độ phủ khác nhau. Không có log chưa chứng minh người học không hoạt động.\',\n        \'\', \'## Bổ sung sentiment sau\' if pending else \'## Nguồn sentiment\',\'\',\n        (\'Cung cấp parquet có cột `comment_id, p_pos, p_neu, p_neg`, mỗi comment một dòng, xác suất tổng bằng 1; \'\n         \'ghi kèm model/revision/ngôn ngữ và kiểm định. Chỉ dự đoán các comment trong `tables/afelo_comment_links.parquet`.\' if pending else\n         \'AFELO đã dùng xác suất sentiment theo comment_id; đường dẫn và SHA256 nằm trong run_manifest.json, \'\n         \'mục finalization. Notebook 07 lưu model/revision, xác suất, cờ truncation/ngôn ngữ và báo cáo riêng.\'),\n        (\'Sau đó gọi \' if pending else \'Đã gọi \')+ \'`finalize_quality(output_root, sentiment_path=..., afelo_mode="sentiment")`. \'\n        \'Hàm chỉ đọc các bảng đã tạo, không quét lại video/bài tập. Mode social_only là lựa chọn riêng, không được bật ngầm.\',\n        \'\', \'## Kiểm tra\',\'\',f\'Đạt {sum(s["validation"].values())}/{len(s["validation"])} kiểm tra cuối, cùng các kiểm tra nền và fixtures \'\n        \'trong validation.json. run_manifest.json giữ SQL, fingerprint nguồn và cấu hình.\',\'\']\n    (out/\'bao_cao_final_quality.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--raw-root\');p.add_argument(\'--model-root\');p.add_argument(\'--time-root\');p.add_argument(\'--output-root\')\n    a=p.parse_args();run_build(a.raw_root,a.model_root,a.time_root,a.output_root)\n'


In [ ]:
"""Run pinned Cardiff sentiment locally, then finalize persisted quality tables.

No video/problem rescan, no fitting on dev/test, no hosted comment inference.
Each prediction is checkpointed by unique preprocessed text, then joined back
to every accepted comment_id. Re-running resumes the same pinned revision.
"""
from __future__ import annotations

import datetime as dt
import hashlib
import importlib.util
import json
import math
import os
from pathlib import Path
import re
import time

os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import duckdb

MODEL_ID = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
PREPROCESS_VERSION = 'cardiff-model-card-mentions-http-v1'
MAX_LENGTH = 512


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def dump_json(path, value):
    p = Path(path)
    temporary = p.with_suffix(p.suffix + '.partial')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    temporary.replace(p)


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def preprocess(text):
    # Exactly the preprocessing published on the selected model card.
    words = []
    for word in text.split(' '):
        word = '@user' if word.startswith('@') and len(word) > 1 else word
        word = 'http' if word.startswith('http') else word
        words.append(word)
    return ' '.join(words)


def resolve_model(output_root, cache_dir, allow_download=True):
    from huggingface_hub import HfApi, snapshot_download
    out = Path(output_root)
    out.mkdir(parents=True, exist_ok=True)
    lock_path = out / 'model_lock.json'
    if lock_path.exists():
        lock = json.loads(lock_path.read_text())
        assert lock['model_id'] == MODEL_ID
    else:
        if not allow_download:
            raise RuntimeError('Model has not been downloaded; run once with allow_download=True.')
        info = HfApi().model_info(MODEL_ID)
        names = {s.rfilename for s in info.siblings}
        weight = 'model.safetensors' if 'model.safetensors' in names else 'pytorch_model.bin'
        wanted = [x for x in ['config.json', 'tokenizer.json', 'tokenizer_config.json',
            'special_tokens_map.json', 'vocab.json', 'merges.txt', weight, 'README.md'] if x in names]
        lock = {'model_id': MODEL_ID, 'revision': info.sha, 'files': wanted}
    path = Path(lock.get('snapshot_path', ''))
    if not path.is_dir() or not all((path / f).is_file() for f in lock['files']):
        path = Path(snapshot_download(MODEL_ID, revision=lock['revision'],
            allow_patterns=lock['files'], cache_dir=str(cache_dir), local_files_only=not allow_download))
    lock['snapshot_path'] = str(path.resolve())
    lock['file_sha256'] = {f: file_hash(path / f) for f in lock['files']}
    dump_json(lock_path, lock)
    return lock


def prepare_comments(con, source, out):
    stage = out / '_stage'
    stage.mkdir(exist_ok=True)
    signature = file_hash(source)
    checkpoint = stage / 'preprocessing.json'
    paths = [stage / 'unique_texts.parquet', stage / 'comment_text_keys.parquet']
    expected = {'source_sha256': signature, 'preprocess_version': PREPROCESS_VERSION}
    if checkpoint.exists() and json.loads(checkpoint.read_text()) == expected and all(p.exists() for p in paths):
        return paths, signature
    rows = con.execute(f'SELECT comment_id,text FROM read_parquet({sql_string(source)}) ORDER BY comment_id').fetchall()
    assert len(rows) == len({r[0] for r in rows}), 'Input comment IDs must be unique'
    unique = {}
    key_json = stage / 'comment_text_keys.jsonl'
    text_json = stage / 'unique_texts.jsonl'
    with key_json.open('w', encoding='utf-8') as keys:
        for comment_id, text in rows:
            assert isinstance(text, str) and text.strip(), 'Input must contain filtered nonempty comments'
            processed = preprocess(text)
            digest = hashlib.sha256(processed.encode('utf-8')).hexdigest()
            if digest in unique:
                assert unique[digest] == processed, 'Text hash collision'
            unique[digest] = processed
            keys.write(json.dumps({'comment_id': comment_id, 'text_hash': digest}) + '\n')
    with text_json.open('w', encoding='utf-8') as texts:
        for digest, text in sorted(unique.items()):
            texts.write(json.dumps({'text_hash': digest, 'model_text': text,
                'has_cjk': bool(re.search(r'[\u3400-\u9fff]', text))}, ensure_ascii=False) + '\n')
    for json_path, parquet_path in zip([text_json, key_json], paths):
        con.execute(f'COPY (SELECT * FROM read_json_auto({sql_string(json_path)})) TO {sql_string(parquet_path)} (FORMAT PARQUET, COMPRESSION ZSTD)')
        json_path.unlink()
    dump_json(checkpoint, expected)
    print(f'Prepared {len(rows):,} comments; {len(unique):,} unique model texts', flush=True)
    return paths, signature


def infer_unique(con, unique_path, out, lock, source_hash, device='auto', threads=4,
                 batch_size=32, token_budget=2048, checkpoint_size=1024):
    import torch
    import transformers
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    torch.set_num_threads(threads)
    torch.manual_seed(0)
    if device == 'auto':
        device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    model_path = lock['snapshot_path']
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
    labels = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
    assert labels == {0: 'negative', 1: 'neutral', 2: 'positive'}, labels
    model.eval().to(device)
    rows = con.execute(f'SELECT text_hash,model_text,has_cjk FROM read_parquet({sql_string(unique_path)}) ORDER BY text_hash').fetchall()
    texts = [r[1] for r in rows]
    encoded = tokenizer(texts, truncation=False, padding=False, verbose=False)['input_ids']
    items = []
    for row, ids in zip(rows, encoded):
        n_tokens = len(ids)
        if n_tokens > MAX_LENGTH:
            ids = tokenizer(row[1], truncation=True, max_length=MAX_LENGTH)['input_ids']
        items.append({'text_hash': row[0], 'has_cjk': row[2], 'n_tokens': n_tokens, 'ids': ids})
    items.sort(key=lambda r: (len(r['ids']), r['text_hash']))
    signature_payload = {'source_sha256': source_hash, 'model_revision': lock['revision'],
        'preprocess_version': PREPROCESS_VERSION, 'max_length': MAX_LENGTH,
        'torch': torch.__version__, 'transformers': transformers.__version__, 'device': device,
        'batch_size': batch_size, 'token_budget': token_budget, 'checkpoint_size': checkpoint_size}
    signature = hashlib.sha256(json.dumps(signature_payload, sort_keys=True).encode()).hexdigest()
    predictions = out / 'text_predictions'
    predictions.mkdir(exist_ok=True)
    info_path = predictions / 'inference_config.json'
    if info_path.exists():
        assert json.loads(info_path.read_text())['signature'] == signature, 'Inference config changed; use a new output directory'
    dump_json(info_path, dict(signature_payload, signature=signature, labels=labels, dtype='float32'))
    started = time.monotonic()
    n_parts = math.ceil(len(items) / checkpoint_size)
    for part in range(n_parts):
        path = predictions / f'part-{part:05d}.parquet'
        chunk = items[part * checkpoint_size:(part + 1) * checkpoint_size]
        if path.exists():
            assert con.execute(f'SELECT count(*) FROM read_parquet({sql_string(path)})').fetchone()[0] == len(chunk)
            print(f'Sentiment checkpoint {part + 1}/{n_parts} reused', flush=True)
            continue
        result = []
        pos = 0
        tick = time.monotonic()
        while pos < len(chunk):
            length = len(chunk[min(pos + batch_size, len(chunk)) - 1]['ids'])
            n = max(1, min(batch_size, token_budget // max(length, 1)))
            batch = chunk[pos:pos + n]
            inputs = tokenizer.pad({'input_ids': [r['ids'] for r in batch]}, padding=True, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.inference_mode():
                probabilities = torch.softmax(model(**inputs).logits.float(), dim=-1).cpu().double().tolist()
            for item, values in zip(batch, probabilities):
                # Normalize float32 rounding into a double probability simplex.
                total = sum(values)
                negative, neutral, positive = [v / total for v in values]
                p = [negative, neutral, positive]
                result.append({'text_hash': item['text_hash'], 'p_neg': negative, 'p_neu': neutral,
                    'p_pos': positive, 'predicted_label': labels[max(range(3), key=lambda i: p[i])],
                    'confidence': max(p), 'entropy': -sum(v * math.log(v) for v in p if v > 0),
                    'n_tokens': item['n_tokens'], 'n_tokens_used': len(item['ids']),
                    'was_truncated': item['n_tokens'] > MAX_LENGTH, 'has_cjk': item['has_cjk']})
            pos += len(batch)
        temporary = path.with_suffix('.jsonl')
        with temporary.open('w') as f:
            for record in result:
                f.write(json.dumps(record) + '\n')
        partial = path.with_suffix('.parquet.partial')
        con.execute(f'COPY (SELECT * FROM read_json_auto({sql_string(temporary)})) TO {sql_string(partial)} (FORMAT PARQUET,COMPRESSION ZSTD)')
        partial.replace(path)
        temporary.unlink()
        print(f'Sentiment {min((part+1)*checkpoint_size,len(items)):,}/{len(items):,}; '
              f'checkpoint {part+1}/{n_parts}, {time.monotonic()-tick:.1f}s, elapsed {time.monotonic()-started:.1f}s', flush=True)
    return predictions, dict(signature_payload, signature=signature, dtype='float32', labels=labels)


def load_finalizer(path):
    if Path(path).is_file():
        spec = importlib.util.spec_from_file_location('quality_finalizer', path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module.finalize_quality, file_hash(path)
    if 'FINALIZER_SOURCE' in globals():
        source = globals()['FINALIZER_SOURCE']
        namespace = {'__name__': 'quality_finalizer', 'ENGINE_SOURCE_SHA256': hashlib.sha256(source.encode()).hexdigest()}
        exec(compile(source, 'embedded_quality_finalizer.py', 'exec'), namespace)
        return namespace['finalize_quality'], namespace['ENGINE_SOURCE_SHA256']
    raise FileNotFoundError(path)


def run_sentiment(quality_root=None, output_root=None, cache_dir=None, engine_path=None,
                  allow_download=True, device='auto', threads=4, finalize=True):
    default_base = Path.cwd() / 'working_13-10' if (Path.cwd() / 'working_13-10').is_dir() else Path.cwd()
    quality = Path(quality_root or default_base / 'final_quality_tables').resolve()
    out = Path(output_root or quality.parent / 'afelo_sentiment_cardiff').resolve()
    cache = Path(cache_dir or os.environ.get('HF_HOME', str(Path.home() / '.cache/huggingface/hub')))
    source = quality / 'tables/afelo_comment_links.parquet'
    assert source.is_file(), source
    out.mkdir(parents=True, exist_ok=True)
    for folder in [out / 'tables', out / '_scratch']:
        folder.mkdir(exist_ok=True)
    success = out / '_SUCCESS.json'
    if success.exists():
        success.unlink()
    lock = resolve_model(out, cache, allow_download)
    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute('SET threads=2')
    con.execute(f'SET temp_directory={sql_string(out / "_scratch")}')
    (unique, keys), source_hash = prepare_comments(con, source, out)
    predictions, inference = infer_unique(con, unique, out, lock, source_hash, device=device, threads=threads)
    for name, path in [('source_comments', source), ('comment_keys', keys), ('text_predictions', predictions / '*.parquet')]:
        con.execute(f'CREATE VIEW {name} AS SELECT * FROM read_parquet({sql_string(path)})')
    result_path = out / 'tables/comment_sentiment.parquet'
    partial = result_path.with_suffix('.parquet.partial')
    con.execute(f'''COPY (SELECT c.comment_id,c.user_id,c.course_id,c.lo_key,c.event_time,
        p.*,{sql_string(MODEL_ID)} model_id,{sql_string(lock['revision'])} model_revision
        FROM source_comments c LEFT JOIN comment_keys k USING(comment_id)
        LEFT JOIN text_predictions p USING(text_hash)) TO {sql_string(partial)} (FORMAT PARQUET,COMPRESSION ZSTD)''')
    partial.replace(result_path)
    con.execute(f'CREATE VIEW sentiment AS SELECT * FROM read_parquet({sql_string(result_path)})')
    checks = {
        'all_source_comments_preserved': con.execute('SELECT (SELECT count(*) FROM source_comments)=(SELECT count(*) FROM sentiment)').fetchone()[0],
        'unique_comment_ids': con.execute('SELECT count(*)=count(DISTINCT comment_id) FROM sentiment').fetchone()[0],
        'no_missing_predictions': con.execute('SELECT count(*)=0 FROM sentiment WHERE p_pos IS NULL OR p_neu IS NULL OR p_neg IS NULL OR text_hash IS NULL').fetchone()[0],
        'valid_probabilities': con.execute('''SELECT count(*)=0 FROM sentiment WHERE NOT isfinite(p_pos) OR NOT isfinite(p_neu) OR NOT isfinite(p_neg)
            OR p_pos<0 OR p_neu<0 OR p_neg<0 OR p_pos>1 OR p_neu>1 OR p_neg>1 OR abs(p_pos+p_neu+p_neg-1)>1e-6''').fetchone()[0],
        'events_inside_user_final_window': con.execute('SELECT count(*)=0 FROM source_comments WHERE event_time<enroll_time OR event_time>course_end').fetchone()[0],
        'prediction_keys_match_source': con.execute('SELECT count(*)=0 FROM source_comments ANTI JOIN sentiment USING(comment_id)').fetchone()[0],
        'preprocessing_fixture': preprocess('@alice hello https://example.com') == '@user hello http',
    }
    assert all(checks.values()), checks
    def records(sql):
        cursor = con.execute(sql)
        names = [x[0] for x in cursor.description]
        return [dict(zip(names, row)) for row in cursor.fetchall()]
    summary = {'comments': con.execute('SELECT count(*) FROM sentiment').fetchone()[0],
        'unique_model_texts': con.execute('SELECT count(*) FROM text_predictions').fetchone()[0],
        'truncated_comments': con.execute('SELECT count(*) FROM sentiment WHERE was_truncated').fetchone()[0],
        'comments_with_cjk': con.execute('SELECT count(*) FROM sentiment WHERE has_cjk').fetchone()[0],
        'label_distribution': records('SELECT predicted_label,count(*) n_comments,avg(confidence) mean_confidence FROM sentiment GROUP BY 1 ORDER BY 1'),
        'checks': checks}
    audit_path = out / 'tables/manual_audit_sample.parquet'
    con.execute(f'''COPY (SELECT * EXCLUDE(sample_rank) FROM (
        SELECT c.text,s.*,row_number() OVER(PARTITION BY predicted_label ORDER BY hash(comment_id)) sample_rank
        FROM sentiment s JOIN source_comments c USING(comment_id)) WHERE sample_rank<=30)
        TO {sql_string(audit_path)} (FORMAT PARQUET,COMPRESSION ZSTD)''')
    con.close()
    import importlib.metadata
    packages = {name: importlib.metadata.version(name) for name in ['torch','transformers','tokenizers','huggingface-hub','duckdb','numpy']}
    manifest = {'created_utc': dt.datetime.now(dt.timezone.utc).isoformat(), 'source': str(source),
        'source_sha256': source_hash, 'model': lock, 'inference': inference, 'packages': packages,
        'sentiment_path': str(result_path), 'sentiment_sha256': file_hash(result_path),
        'script_sha256': globals().get('SENTIMENT_SOURCE_SHA256') or file_hash(__file__),
        'preprocessing': PREPROCESS_VERSION, 'training': 'Frozen pretrained model, no fitting or threshold tuning on dataset',
        'limitations': ['English Twitter sentiment model on translated educational comments; no domain accuracy claim without annotated evaluation.',
            'Mixed-language/CJK comments are retained and flagged; sentiment remains a proxy.',
            'Long comments use the first 512 tokens including special tokens; truncation is flagged.',
            'Source filter removes empty/exact duplicate comments; it does not identify all off-topic/UI/check-in comments.',
            'No-comment chapters keep sentiment null; AFELO can still use observed social participation components.']}
    if finalize:
        backup = out / 'pre_sentiment_state'
        backup.mkdir(exist_ok=True)
        for name in ['run_manifest.json','summary.json','validation.json','_SUCCESS.json','bao_cao_final_quality.md']:
            p = quality / name
            if p.exists() and not (backup / name).exists():
                (backup / name).write_bytes(p.read_bytes())
        finalizer, engine_hash = load_finalizer(engine_path or quality.parent / '06_build_final_quality_tables.py')
        summary['quality'] = finalizer(quality, sentiment_path=result_path, afelo_mode='sentiment')
        manifest['finalizer_source_sha256'] = engine_hash
        quality_manifest = json.loads((quality / 'run_manifest.json').read_text())
        quality_manifest['finalization']['sentiment_model'] = {'model_id': MODEL_ID, 'revision': lock['revision'],
            'sentiment_manifest': str(out / 'run_manifest.json')}
        dump_json(quality / 'run_manifest.json', quality_manifest)
        dump_json(quality / '_SUCCESS.json', {'afelo_mode': 'sentiment', 'manifest_sha256': file_hash(quality / 'run_manifest.json')})
    dump_json(out / 'summary.json', summary)
    dump_json(out / 'run_manifest.json', manifest)
    dump_json(out / 'validation.json', checks)
    dump_json(success, {'manifest_sha256': file_hash(out / 'run_manifest.json'), 'finalized': finalize})
    write_sentiment_report(out, quality, summary, lock)
    print('SENTIMENT_COMPLETE', json.dumps(summary, ensure_ascii=False, default=str), flush=True)
    return summary


def write_sentiment_report(out, quality, summary, lock):
    fmt = lambda n: f'{n:,}'.replace(',', '.')
    lines = ['# AFELO với Cardiff sentiment', '',
        f'Model: [{MODEL_ID}](https://huggingface.co/{MODEL_ID}), revision `{lock["revision"]}`. '
        'Chạy tại máy, frozen weights, không huấn luyện hoặc fit tham số trên comment của dataset.', '',
        f'- Đã suy luận và ghép lại đủ **{fmt(summary["comments"])} comment** từ '
        f'**{fmt(summary["unique_model_texts"])} nội dung sau tiền xử lý**.',
        f'- {fmt(summary["truncated_comments"])} comment dài hơn 512 token: lấy phần đầu và gắn cờ `was_truncated`.',
        f'- {fmt(summary["comments_with_cjk"])} comment có ký tự CJK: giữ và gắn cờ `has_cjk`.',
        '- Đầu vào đã lọc theo `[enroll_time, course_end]`, loại text rỗng và bản ghi trùng chính xác. '
        'Đây chưa phải bộ lọc ngữ nghĩa cho comment học tập.', '',
        '| Nhãn sentiment dự đoán | Số comment |', '|---|---:|']
    lines += [f'| {r["predicted_label"]} | {fmt(r["n_comments"])} |' for r in summary['label_distribution']]
    lines += ['', '## Công thức và dữ liệu thiếu', '',
        '`sentiment_valence = ((sum(p_pos) - sum(p_neg)) / n_sentiment_comments + 1) / 2`.', '',
        '`AFELO_proxy = mean_non_null(comment_participation, reply_participation, response_received_rate, sentiment_valence)`.', '',
        'Không có comment thì sentiment và response rate là null. Các thành phần tham gia xã hội còn lại vẫn được tính '
        'theo quy ước notebook 06; AFELO vì thế không đồng nghĩa chỉ với cảm xúc dự đoán. '
        'Chương hoàn toàn chưa quan sát không có dòng chapter và không được tự gán chất lượng kém.', '',
        'Model học từ tweet tiếng Anh; comment dịch trong giáo dục có khác biệt miền dữ liệu. Xác suất/confidence '
        'không phải độ chính xác đã kiểm định. `manual_audit_sample.parquet` chứa 30 mẫu mỗi nhãn để kiểm tra thủ công. '
        'AFELO vẫn là proxy do nguồn không có like/dislike/rating gốc.', '']
    if 'quality' in summary:
        q = summary['quality']
        lines += ['## Bảng cuối kỳ', '',
            f'Giữ đủ **{fmt(q["user_course_rows"])} user–course** và **{fmt(q["chapter_rows"])} chương quan sát**. '
            'Các bảng được cập nhật bằng LEFT JOIN; không bỏ dòng do thiếu chỉ số.', '',
            '| Trạng thái | Số user–course |', '|---|---:|']
        lines += [f'| {r["label_status"]} | {fmt(r["n_rows"])} |' for r in q['label_status']]
        lines += ['', '| Split | Lớp | Số user–course |', '|---|---|---:|']
        lines += [f'| {r["split"]} | {r["y"]} | {fmt(r["n_rows"])} |' for r in q['class_distribution']]
        lines += ['', 'q chỉ có khi tất cả chương quan sát có đủ ba chỉ số; q là trung bình LOQV từng chương. '
            'Các cặp chưa quan sát hoặc thiếu COELO/ACELO vẫn giữ q/y null.', '',
            'Bảng cuối: `../final_quality_tables/tables/user_course_final/*.parquet` (32 file, một bảng logic). '
            'Ba bảng thành phần và chapter_quality trong cùng thư mục tables đã được cập nhật.', '',
            'Các bảng cuối kỳ phục vụ nhãn/kiểm toán. Feature P1–P4 phải tính lại đến đúng cutoff; '
            'không đưa feature cuối kỳ hoặc chỉ số tạo nhãn vào X. Split course hiện vẫn sơ bộ và còn chồng thời gian '
            'như báo cáo notebook 06; chạy sentiment không giải quyết giới hạn này.', '']
    lines += ['## Chạy lại', '',
        'Notebook 07 / script 07 tiếp tục từ checkpoint và pin revision trong model_lock.json. '
        'Notebook 06 dựng dữ liệu nền ở mode deferred; nếu chạy lại notebook 06, cần chạy tiếp notebook 07 '
        'để điền lại sentiment và nhãn. Không chạy lại dữ liệu thô chỉ để đổi sentiment.', '']
    (out / 'bao_cao_afelo_sentiment.md').write_text('\n'.join(lines), encoding='utf-8')


SENTIMENT_SOURCE_SHA256 = 'c8c32661b2e91c81d43a91fc519394c2b6bb112cded381146090e36f0ca6150f'
print("Embedded sentiment engine:", SENTIMENT_SOURCE_SHA256)


## Chạy hoặc đọc lại kết quả đã hoàn tất

Khi có `_SUCCESS` hợp lệ, cùng SHA256 đầu vào, xác suất, source engine và cùng manifest cuối kỳ, cell đọc lại kết quả. Nếu chưa hoàn tất, tiếp tục suy luận từ checkpoint. Notebook 06 khởi tạo mode deferred; sau khi chạy lại 06 cần chạy tiếp 07.


In [ ]:
manifest_path = OUTPUT_ROOT / 'run_manifest.json'
success_path = OUTPUT_ROOT / '_SUCCESS.json'
can_reuse = False
if REUSE_COMPLETE and manifest_path.exists() and success_path.exists() and (QUALITY_ROOT / '_SUCCESS.json').exists():
    previous = json.loads(manifest_path.read_text())
    success = json.loads(success_path.read_text())
    quality_manifest = json.loads((QUALITY_ROOT / 'run_manifest.json').read_text())
    quality_success = json.loads((QUALITY_ROOT / '_SUCCESS.json').read_text())
    probabilities = OUTPUT_ROOT / 'tables/comment_sentiment.parquet'
    can_reuse = (success.get('finalized') is True
        and success['manifest_sha256'] == file_hash(manifest_path)
        and previous['script_sha256'] == SENTIMENT_SOURCE_SHA256
        and previous['source_sha256'] == file_hash(QUALITY_ROOT / 'tables/afelo_comment_links.parquet')
        and probabilities.exists()
        and previous['sentiment_sha256'] == file_hash(probabilities)
        and quality_manifest.get('finalization', {}).get('afelo_mode') == 'sentiment'
        and quality_manifest['finalization']['sentiment_sha256'] == previous['sentiment_sha256']
        and quality_success['manifest_sha256'] == file_hash(QUALITY_ROOT / 'run_manifest.json'))
if can_reuse:
    RESULT = json.loads((OUTPUT_ROOT / 'summary.json').read_text())
    print('Reused completed run: input, predictions, engine and finalization hashes verified.')
else:
    RESULT = run_sentiment(QUALITY_ROOT, OUTPUT_ROOT, CACHE_DIR, ENGINE_PATH,
                          allow_download=ALLOW_DOWNLOAD, device=DEVICE, threads=THREADS)
print(json.dumps(RESULT, ensure_ascii=False, indent=2))


## Kiểm tra bảng đã lưu

Mỗi comment đầu vào phải có đúng một dự đoán. Bảng chương và bảng user–course tiếp tục giữ mọi đối tượng đã có ở notebook 06; nhãn chỉ có khi mọi chương quan sát đủ ba chỉ số.


In [ ]:
con = duckdb.connect()
predictions = OUTPUT_ROOT / 'tables/comment_sentiment.parquet'
chapters = QUALITY_ROOT / 'tables/chapter_quality.parquet'
final_table = QUALITY_ROOT / 'tables/user_course_final/*.parquet'
print('Sentiment:', con.execute(f'SELECT count(*),count(DISTINCT comment_id) FROM read_parquet({sql_string(predictions)})').fetchone())
print('Chapter status:', con.execute(f'SELECT chapter_status,count(*) FROM read_parquet({sql_string(chapters)}) GROUP BY 1').fetchall())
print('Final status:', con.execute(f'SELECT label_status,count(*) FROM read_parquet({sql_string(final_table)}) GROUP BY 1').fetchall())
print('Classes:', con.execute(f"SELECT y,count(*) FROM read_parquet({sql_string(final_table)}) WHERE y IS NOT NULL GROUP BY 1 ORDER BY 1").fetchall())
assert all(RESULT['checks'].values())
assert all(RESULT['quality']['validation'].values())
print('All stored checks passed.')
con.close()


## Chẩn đoán phân bố nhãn trước khi huấn luyện

Giữ nguyên công thức và ngưỡng đã thống nhất, tính độ phủ sentiment và phân bố q/y. AFELO bằng 0 do không có comment/reply là hệ quả của các chỉ báo tham gia bằng 0; đây không phải dự đoán sentiment tiêu cực. Nếu AFELO=0 và đủ ba chỉ số, LOQV ≥ 1/√3 ≈ 0,577, nên chương đó thuộc lớp A theo ngưỡng hiện tại. Không tự đổi ngưỡng bằng phân vị dev/test.


In [ ]:
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute('SET threads=2')
tables = QUALITY_ROOT / 'tables'
queries = {
    'chapter_afelo': f"SELECT count(*) n_chapters,count(*) FILTER(WHERE n_sentiment_comments>0) n_with_sentiment,count(*) FILTER(WHERE AFELO_proxy=0) n_afelo_zero,count(*) FILTER(WHERE n_comments_filtered=0 AND n_replies_authored=0) n_no_comments_no_replies,avg(AFELO_proxy) mean_afelo,quantile_cont(AFELO_proxy,[0.5,0.9,0.95,0.99]) afelo_quantiles FROM read_parquet('{tables}/afelo_components.parquet')",
    'chapter_indices': f"SELECT avg(COELO) mean_coelo,avg(AFELO_proxy) mean_afelo,avg(ACELO) mean_acelo,count(*) FILTER(WHERE LOQV<0.2) n_e_chapters,count(*) FILTER(WHERE LOQV>=0.2 AND LOQV<0.4) n_g_chapters,count(*) FILTER(WHERE LOQV>=0.4) n_a_chapters FROM read_parquet('{tables}/chapter_quality.parquet')",
    'user_course_q': f"SELECT min(q) min_q,max(q) max_q,quantile_cont(q,[0.01,0.05,0.25,0.5,0.75,0.95,0.99]) q_quantiles,avg(coelo_mean_chapter) mean_coelo,avg(afelo_proxy_mean_chapter) mean_afelo,avg(acelo_mean_chapter) mean_acelo FROM read_parquet('{tables}/user_course_final/*.parquet') WHERE y IS NOT NULL",
    'user_course_classes': f"SELECT y,count(*) n_rows FROM read_parquet('{tables}/user_course_final/*.parquet') WHERE y IS NOT NULL GROUP BY 1 ORDER BY 1",
}
diagnostics = {}
for name, sql in queries.items():
    cursor = con.execute(sql)
    names = [x[0] for x in cursor.description]
    diagnostics[name] = [dict(zip(names, row)) for row in cursor.fetchall()]
con.close()
dump_json(OUTPUT_ROOT / 'label_diagnostics.json', diagnostics)
a = diagnostics['chapter_afelo'][0]
q = diagnostics['user_course_q'][0]
classes = {r['y']: r['n_rows'] for r in diagnostics['user_course_classes']}
n_labeled = sum(classes.values())
fmt = lambda n: f'{n:,}'.replace(',', '.')
lines = ['# Chẩn đoán nhãn sau khi tính AFELO', '',
    f'Đã tính đúng công thức hiện tại cho {fmt(n_labeled)} user–course. '
    'Phân bố kết quả chưa phù hợp để coi như một bộ nhãn ba lớp ổn định.', '',
    '| Lớp | Số user–course | Tỷ lệ |', '|---|---:|---:|']
for label in ['E', 'G', 'A']:
    n = classes.get(label, 0)
    lines.append(f'| {label} | {fmt(n)} | {n/n_labeled*100:.4f}% |')
lines += ['', '## Độ phủ và tác động của công thức', '',
    f'- {fmt(a["n_with_sentiment"])}/{fmt(a["n_chapters"])} chương quan sát có sentiment '
    f'({a["n_with_sentiment"]/a["n_chapters"]*100:.2f}%).',
    f'- {fmt(a["n_afelo_zero"])} chương có AFELO_proxy=0 '
    f'({a["n_afelo_zero"]/a["n_chapters"]*100:.2f}%), trùng với số chương không có comment và reply do người học viết.',
    f'- AFELO trung bình chương: {a["mean_afelo"]:.4f}. '
    f'q của các cặp có nhãn nằm trong [{q["min_q"]:.4f}, {q["max_q"]:.4f}].', '',
    'Với chương không có comment/reply, hai chỉ báo tham gia bằng 0, sentiment và response rate null, '
    'nên mean_non_null cho AFELO_proxy bằng 0. **Model không dự đoán cảm xúc tiêu cực cho những chương này.** '
    'Nếu chương đủ COELO/ACELO thì LOQV ≥ sqrt(1/3) ≈ 0,577 và bắt buộc rơi vào lớp A theo ngưỡng 0,4. '
    'Đây là một cơ chế đẩy nhãn về A, xuất phát từ độ phủ nguồn xã hội và cách biểu diễn vắng tương tác.', '',
    f'Có {diagnostics["chapter_indices"][0]["n_e_chapters"]} chương riêng lẻ đạt E; q lấy trung bình mọi chương quan sát. '
    'Điều này không chứng minh người học có chất lượng thấp hoặc sentiment model sai.', '',
    '## Cần quyết định trước khi dùng nhãn cho mô hình', '',
    'Rà lại cách diễn giải AFELO ở chương vắng dữ liệu xã hội và kiểm định proxy/nhãn trên mẫu thủ công. '
    'Nếu chọn cách xử lý thiếu khác, cần ghi thành một phiên bản công thức riêng và đánh giá độ nhạy. '
    'Chưa thay đổi công thức/ngưỡng trong lần chạy này; không chọn ngưỡng từ phân bố dev/test chỉ để cân bằng lớp. '
    'Cần xử lý thêm vấn đề course chồng thời gian của split hiện tại khi thiết kế thí nghiệm.', '']
(OUTPUT_ROOT / 'bao_cao_phan_bo_nhan.md').write_text('\n'.join(lines), encoding='utf-8')
print(json.dumps(diagnostics, ensure_ascii=False, indent=2))
print('Diagnostic report:', OUTPUT_ROOT / 'bao_cao_phan_bo_nhan.md')

main_report = OUTPUT_ROOT / 'bao_cao_afelo_sentiment.md'
if main_report.exists():
    marker = '\n## Chẩn đoán nhãn sau chạy'
    content = main_report.read_text().split(marker, 1)[0]
    content += marker + '\n\n'
    content += (f'Phân bố user–course: E={classes.get("E",0):,}, G={classes.get("G",0):,}, '
                f'A={classes.get("A",0):,} ({classes.get("A",0)/n_labeled*100:.4f}%). '
                'Cần rà lại độ phủ xã hội và cách diễn giải AFELO trước khi dùng như bộ nhãn ba lớp. '
                'Xem [chẩn đoán phân bố nhãn](bao_cao_phan_bo_nhan.md).\n')
    main_report.write_text(content, encoding='utf-8')


## Sản phẩm và giới hạn

- `afelo_sentiment_cardiff/tables/comment_sentiment.parquet`: xác suất, nhãn dự đoán, confidence, token counts, cờ truncation/CJK và model revision theo comment_id.
- `afelo_sentiment_cardiff/tables/manual_audit_sample.parquet`: 30 comment mỗi nhãn để đánh giá thủ công; chưa có nhãn người gán.
- `final_quality_tables/tables/afelo_components.parquet`, `chapter_quality.parquet` và `user_course_final/*.parquet`: cập nhật AFELO/LOQV/q/y.
- Báo cáo sentiment, summary, manifest, model lock và validation lưu cùng output. `chapter_details` giữ mọi feature con theo chương.

Không có log chưa chứng minh không hoạt động. AFELO của chương không có comment dựa vào thành phần xã hội, không phải sentiment. Các cặp thiếu COELO/ACELO vẫn chưa có nhãn. Các bảng cuối kỳ chỉ dùng làm nhãn/kiểm toán; X của P1–P4 phải tính riêng đến cutoff. Split course sơ bộ vẫn có chồng thời gian, xem báo cáo 06.
